## Data Reading

### CSV Files

In [64]:
import os
import sys

# Force the kernel to use your Homebrew Java 17 path
os.environ["JAVA_HOME"] = (
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
)

# Tell PySpark exactly which Python executable to target (prevents system mismatches)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [65]:
from pyspark.sql import SparkSession

In [66]:
spark = SparkSession.builder.appName("Citibike Analysis").getOrCreate()

In [67]:
main_schema = """
    ride_id string, 
    rideable_type string,
    started_at timestamp,
    ended_at timestamp,
    start_station_name string, 
    start_station_id string,
    end_station_name string,
    end_station_id string,
    start_lat double,
    start_lng double,
    end_lat double,
    end_lng double,
    member_casual string
"""

trips = (
    spark.read.format("csv")
    .schema(main_schema)
    .option("header", True)
    .load("/Users/pius/Projects/analytics-engineering/Spark/data/JC*.csv")
)

26/07/02 10:21:52 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /Users/pius/Projects/analytics-engineering/Spark/data/JC*.csv.
java.io.FileNotFoundException: File /Users/pius/Projects/analytics-engineering/Spark/data/JC*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSo

In [68]:
print(f"Total rows loaded: {trips.count()}")

Total rows loaded: 4700530


In [69]:
# Or

# trips = spark.read.csv(
#     "/home/piusm/dev-projects/analytics-engineering/Spark/data/JC*.csv",
#     inferSchema=True,
#     header=True,
# )

#### Alternative way of defining a schema

In [70]:
# my_struct_schema = StructType(
#     [
#         StructField("ride_id", StringType(), True),
#         StructField("rideable_type", StringType(), True),
#         StructField("started_at", TimestampType(), True),
#         StructField("ended_at", TimestampType(), True),
#         StructField("start_station_name", StringType(), True),
#         StructField("start_station_id", StringType(), False),
#         StructField("end_station_name", StringType(), True),
#         StructField("end_station_id", StringType(), True),
#         StructField("start_lat", DoubleType(), True),
#         StructField("start_lng", DoubleType(), True),
#         StructField("end_lat", DoubleType(), True),
#         StructField("end_lng", DoubleType(), True),
#         StructField("member_casual", StringType(), True),
#     ]
# )

In [71]:
trips.show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+
|172DBBFC733F03CE|electric_bike|2024-10-10 14:54:...|2024-10-10 15:04:...|         Oakland Ave|           JC022|Stevens - River T...|         HB602|       40.7376037|       -74.0524783|40.74313282710551|-74.02698867022991|       member|
|D20BBA4860FE736C|electric_bike|2024-10-03 19:20:...

In [72]:
trips.count()

4700530

In [73]:
trips.limit(10).toPandas()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,172DBBFC733F03CE,electric_bike,2024-10-10 14:54:24.572,2024-10-10 15:04:07.657,Oakland Ave,JC022,Stevens - River Ter & 6 St,HB602,40.737604,-74.052478,40.743133,-74.026989,member
1,D20BBA4860FE736C,electric_bike,2024-10-03 19:20:21.215,2024-10-03 19:31:46.511,Oakland Ave,JC022,Stevens - River Ter & 6 St,HB602,40.737604,-74.052478,40.743133,-74.026989,casual
2,86F89348995D0E6E,classic_bike,2024-10-20 12:14:56.318,2024-10-20 12:28:32.053,Oakland Ave,JC022,South Waterfront Walkway - Sinatra Dr & 1 St,HB103,40.737604,-74.052478,40.736982,-74.027781,casual
3,AA55A717B7EC1D10,classic_bike,2024-10-20 14:40:15.227,2024-10-20 14:55:39.100,Oakland Ave,JC022,Columbus Drive,JC014,40.737604,-74.052478,40.718355,-74.038914,member
4,C72953D91E986DA7,classic_bike,2024-10-20 08:37:03.280,2024-10-20 08:42:24.770,Brunswick & 6th,JC081,Washington St,JC098,40.726012,-74.050389,40.724294,-74.035483,member
5,23A1827EA03A9AC2,electric_bike,2024-10-28 19:20:28.668,2024-10-28 19:25:28.448,Oakland Ave,JC022,Hoboken Ave at Monmouth St,JC105,40.737604,-74.052478,40.735208,-74.046964,member
6,6C0E882AE20AC640,electric_bike,2024-10-08 10:41:37.926,2024-10-08 10:44:23.533,Pershing Field,JC024,Leonard Gordon Park,JC080,40.742677,-74.051789,40.745910,-74.057271,member
7,FC4AEE485D39016D,electric_bike,2024-10-25 21:02:40.878,2024-10-25 21:11:00.117,Pershing Field,JC024,Leonard Gordon Park,JC080,40.742677,-74.051789,40.745910,-74.057271,member
8,3E4D96936A8660C6,electric_bike,2024-10-08 12:17:23.948,2024-10-08 12:20:47.035,City Hall - Washington St & 1 St,HB105,Stevens - River Ter & 6 St,HB602,40.737360,-74.030970,40.743133,-74.026989,member
9,9F2FBA8132468A3B,electric_bike,2024-10-28 11:00:40.004,2024-10-28 11:03:12.823,City Hall - Washington St & 1 St,HB105,Stevens - River Ter & 6 St,HB602,40.737360,-74.030970,40.743133,-74.026989,member


### JSON Files

In [74]:
json_file = (
    spark.read.format("json")
    .option("inferSchema", True)
    .option("header", True)
    .option("multiLine", False)
    .load("/Users/pius/Projects/analytics-engineering/Spark/data/drivers.json")
)

In [75]:
json_file.limit(10).toPandas()

,code,dob,driverId,driverRef,name,nationality,number,url
0,HAM,1985-01-07,1,hamilton,"(Lewis, Hamilton)",British,44,http://en.wikipedia.org/wiki/Lewis_Hamilton
1,HEI,1977-05-10,2,heidfeld,"(Nick, Heidfeld)",German,\N,http://en.wikipedia.org/wiki/Nick_Heidfeld
2,ROS,1985-06-27,3,rosberg,"(Nico, Rosberg)",German,6,http://en.wikipedia.org/wiki/Nico_Rosberg
3,ALO,1981-07-29,4,alonso,"(Fernando, Alonso)",Spanish,14,http://en.wikipedia.org/wiki/Fernando_Alonso
4,KOV,1981-10-19,5,kovalainen,"(Heikki, Kovalainen)",Finnish,\N,http://en.wikipedia.org/wiki/Heikki_Kovalainen
5,NAK,1985-01-11,6,nakajima,"(Kazuki, Nakajima)",Japanese,\N,http://en.wikipedia.org/wiki/Kazuki_Nakajima
6,BOU,1979-02-28,7,bourdais,"(Sébastien, Bourdais)",French,\N,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bo...
7,RAI,1979-10-17,8,raikkonen,"(Kimi, Räikkönen)",Finnish,7,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C...
8,KUB,1984-12-07,9,kubica,"(Robert, Kubica)",Polish,88,http://en.wikipedia.org/wiki/Robert_Kubica
9,GLO,1982-03-18,10,glock,"(Timo, Glock)",German,\N,http://en.wikipedia.org/wiki/Timo_Glock


### Schema Definition

In [76]:
trips.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)



In [77]:
trips.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)



#### StructType() - Programmatic way to define schema

In [78]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

### Data Transformation with PySpark

In [79]:
trips.select("ride_id", "rideable_type").show()

+----------------+-------------+
|         ride_id|rideable_type|
+----------------+-------------+
|172DBBFC733F03CE|electric_bike|
|D20BBA4860FE736C|electric_bike|
|86F89348995D0E6E| classic_bike|
|AA55A717B7EC1D10| classic_bike|
|C72953D91E986DA7| classic_bike|
|23A1827EA03A9AC2|electric_bike|
|6C0E882AE20AC640|electric_bike|
|FC4AEE485D39016D|electric_bike|
|3E4D96936A8660C6|electric_bike|
|9F2FBA8132468A3B|electric_bike|
|B71B07C28733404F|electric_bike|
|2EE1AB7AFEFB4CBB| classic_bike|
|A7493DA7AFDDBC44| classic_bike|
|7A956542D29EA30E| classic_bike|
|157EFB12EA94DA9A|electric_bike|
|3A54C7F77E03FB73| classic_bike|
|94374385A5951BEE| classic_bike|
|5E9040B2EF62AE11|electric_bike|
|99D204D580A9BD0C|electric_bike|
|7C3E2DA95C52FDFB|electric_bike|
+----------------+-------------+
only showing top 20 rows


In [80]:
trips.select(col("ride_id"), col("rideable_type")).show()

+----------------+-------------+
|         ride_id|rideable_type|
+----------------+-------------+
|172DBBFC733F03CE|electric_bike|
|D20BBA4860FE736C|electric_bike|
|86F89348995D0E6E| classic_bike|
|AA55A717B7EC1D10| classic_bike|
|C72953D91E986DA7| classic_bike|
|23A1827EA03A9AC2|electric_bike|
|6C0E882AE20AC640|electric_bike|
|FC4AEE485D39016D|electric_bike|
|3E4D96936A8660C6|electric_bike|
|9F2FBA8132468A3B|electric_bike|
|B71B07C28733404F|electric_bike|
|2EE1AB7AFEFB4CBB| classic_bike|
|A7493DA7AFDDBC44| classic_bike|
|7A956542D29EA30E| classic_bike|
|157EFB12EA94DA9A|electric_bike|
|3A54C7F77E03FB73| classic_bike|
|94374385A5951BEE| classic_bike|
|5E9040B2EF62AE11|electric_bike|
|99D204D580A9BD0C|electric_bike|
|7C3E2DA95C52FDFB|electric_bike|
+----------------+-------------+
only showing top 20 rows


#### Using ALIAS

In [81]:
trips.select(col("ride_id").alias("rideId")).show()

+----------------+
|          rideId|
+----------------+
|172DBBFC733F03CE|
|D20BBA4860FE736C|
|86F89348995D0E6E|
|AA55A717B7EC1D10|
|C72953D91E986DA7|
|23A1827EA03A9AC2|
|6C0E882AE20AC640|
|FC4AEE485D39016D|
|3E4D96936A8660C6|
|9F2FBA8132468A3B|
|B71B07C28733404F|
|2EE1AB7AFEFB4CBB|
|A7493DA7AFDDBC44|
|7A956542D29EA30E|
|157EFB12EA94DA9A|
|3A54C7F77E03FB73|
|94374385A5951BEE|
|5E9040B2EF62AE11|
|99D204D580A9BD0C|
|7C3E2DA95C52FDFB|
+----------------+
only showing top 20 rows


### Filtering Data

#### Scenario 1

In [82]:
trips.filter(col("member_casual") == "casual").show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+------------------+------------------+-----------------+------------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|         start_lat|         start_lng|          end_lat|           end_lng|member_casual|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+------------------+------------------+-----------------+------------------+-------------+
|D20BBA4860FE736C|electric_bike|2024-10-03 19:20:...|2024-10-03 19:31:...|         Oakland Ave|           JC022|Stevens - River T...|         HB602|        40.7376037|       -74.0524783|40.74313282710551|-74.02698867022991|       casual|
|86F89348995D0E6E| classic_bike|2024-10-20 12:14

#### Scenario 2


In [83]:
trips.filter(
    (col("member_casual") == "casual") & (col("rideable_type") == "docked_bike")
).show()

+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+---------+----------+-------------+
|         ride_id|rideable_type|         started_at|           ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|start_lat| start_lng|  end_lat|   end_lng|member_casual|
+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+---------+----------+-------------+
|5E8C8B7DFFA9C73E|  docked_bike|2022-08-20 01:17:27|2022-08-20 02:01:02|South Waterfront ...|           HB103|Hoboken Terminal ...|         HB101|40.736982|-74.027781|40.735938|-74.030305|       casual|
|0A0EFF1E90C8710F|  docked_bike|2022-08-06 20:06:16|2022-08-06 20:18:10|South Waterfront ...|           HB103|Mama Johnson Fiel...|         HB404|40.736982|-74.027781| 40.74314|-74.040041|

#### Scenario 3

In [84]:
trips.filter((col("ride_id").isNull())).show()

+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+
|ride_id|rideable_type|started_at|ended_at|start_station_name|start_station_id|end_station_name|end_station_id|start_lat|start_lng|end_lat|end_lng|member_casual|
+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+
+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+



In [85]:
trips.filter((col("start_station_id").isNull()) & (col("ride_id").isNull())).show()

+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+
|ride_id|rideable_type|started_at|ended_at|start_station_name|start_station_id|end_station_name|end_station_id|start_lat|start_lng|end_lat|end_lng|member_casual|
+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+
+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+



In [86]:
trips.where((col("start_station_name") == "Warren St")).show()

+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+----------+------------+-----------------+------------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|start_station_name|start_station_id|    end_station_name|end_station_id| start_lat|   start_lng|          end_lat|           end_lng|member_casual|
+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+----------+------------+-----------------+------------------+-------------+
|143732490A22F95A| classic_bike|2024-10-26 09:54:...|2024-10-26 10:12:...|         Warren St|           JC006|Stevens - River T...|         HB602|40.7211236|-74.03805095|40.74313282710551|-74.02698867022991|       member|
|23BDC5B0B86C99B4| classic_bike|2024-10-26 10:08:...|2024-10-26 10:49:...|         Warren St|           JC006|St

### Handling Null Values

PySpark provides several functions to handle null values in DataFrames, including `fillna()`, `dropna()`, and `replace()`. These functions allow you to fill, drop, or replace null values in your data as needed.

#### dropna()

When using .`dropna()`, you can specify the `how` parameter to determine how rows with null values should be dropped. The options for the `how` parameter are:

- `any`: Drops a row if it contains any null values (default behavior).
- `all`: Drops a row only if all values in that row are null.

In [87]:
trips = trips.dropna(subset=["ride_id", "start_lat", "end_lat", "start_lng", "end_lng"])

In [88]:
trips.count()

4689909

#### withColumnRenamed()

In [89]:
trips.withColumnRenamed("ride_id", "rideId").show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+
|          rideId|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+
|172DBBFC733F03CE|electric_bike|2024-10-10 14:54:...|2024-10-10 15:04:...|         Oakland Ave|           JC022|Stevens - River T...|         HB602|       40.7376037|       -74.0524783|40.74313282710551|-74.02698867022991|       member|
|D20BBA4860FE736C|electric_bike|2024-10-03 19:20:...

#### withColumn() - Adding a new column to the DataFrame

In [90]:
trips = trips.withColumn(
    "rideDurationMinutes",
    round((col("ended_at").cast("long") - col("started_at").cast("long")) / 60, 2),
)

trips = trips.withColumn(
    "rideDurationMinutes",
    when(col("rideDurationMinutes") < 0, col("rideDurationMinutes") + 60).otherwise(
        col("rideDurationMinutes")
    ),
)

trips = trips.withColumn("rideDurationHours", round(col("rideDurationMinutes") / 60, 2))

In [91]:
trips.show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+
|172DBBFC733F03CE|electric_bike|2024-10-10 14:54:...|2024-10-10 15:04:...|         Oakland Ave|           JC022|Stevens - River T...|         HB602|       40.7376037|       -7

In [92]:
R = 6371.0

lat1 = radians(col("start_lat"))
lat2 = radians(col("end_lat"))
lng1 = radians(col("start_lng"))
lng2 = radians(col("end_lng"))

dlat = lat2 - lat1
dlng = lng2 - lng1

a = pow(sin(dlat / 2.0), 2) + cos(lat1) * cos(lat2) * pow(sin(dlng / 2.0), 2)

c = 2 * asin(sqrt(a))

distance_formula = round(R * c, 2)

trips = trips.withColumn("rideDistanceKM", distance_formula).withColumn(
    "rideDistanceM", distance_formula * 1000
)

In [93]:
trips.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- rideDurationMinutes: double (nullable = true)
 |-- rideDurationHours: double (nullable = true)
 |-- rideDistanceKM: double (nullable = true)
 |-- rideDistanceM: double (nullable = true)



In [94]:
trips.filter((col("rideDistanceM") == 0)).show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|E12C216DABC5E494| classic_bike|2024-10-31 11:38:...|2024-10-31 11:41:...|       Washing

In [95]:
trips.filter((col("rideDistanceM") == 0)).count()

261413

#### Type Casting

In [96]:
trips.withColumn("ride_id", col("ride_id").cast(StringType()))

DataFrame[ride_id: string, rideable_type: string, started_at: timestamp, ended_at: timestamp, start_station_name: string, start_station_id: string, end_station_name: string, end_station_id: string, start_lat: double, start_lng: double, end_lat: double, end_lng: double, member_casual: string, rideDurationMinutes: double, rideDurationHours: double, rideDistanceKM: double, rideDistanceM: double]

#### Sort and OrderBy

In [97]:
trips.sort((col("rideDurationMinutes").desc())).show()

+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|         started_at|           ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|start_lat| start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|AFF1F450B44EF59E|  docked_bike|2021-07-31 19:36:50|2021-12-08 16:38:19|South Waterfront ...|           HB103|Grand Concourse &...|       8303

In [98]:
trips.sort((col("rideDurationMinutes").asc())).show()

+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+------------------+------------------+------------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|         started_at|           ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|         start_lat|         start_lng|           end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+------------------+------------------+------------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|5471CCE179A9960F| classic_bike|2023-05-03 12:26:39|2023-05-03 12:26:39|             Hil

In [99]:
trips.sort(
    ["rideDurationMinutes", "rideDistanceKM"], ascending=[0, 0]
).show()  # 0, 0 means ascending = False - boolean

+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|         started_at|           ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|start_lat| start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|AFF1F450B44EF59E|  docked_bike|2021-07-31 19:36:50|2021-12-08 16:38:19|South Waterfront ...|           HB103|Grand Concourse &...|       8303

In [100]:
trips.sort(["rideDurationMinutes", "rideDistanceKM"], ascending=[0, 1]).show()

+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|         started_at|           ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|start_lat| start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+-------------------+-------------------+--------------------+----------------+--------------------+--------------+---------+----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|AFF1F450B44EF59E|  docked_bike|2021-07-31 19:36:50|2021-12-08 16:38:19|South Waterfront ...|           HB103|Grand Concourse &...|       8303

#### Limit

In PySpark, the `DataFrame.limit(num)` method restricts the result count of a DataFrame to a specified number of rows.

It is a __lazy transformation__, meaning that it does not immediately execute the operation but instead builds a logical plan that will be executed when an action is called such as `show()`, `collect()`, or `write()`.

In [101]:
trips.limit(10).show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|172DBBFC733F03CE|electric_bike|2024-10-10 14:54:...|2024-10-10 15:04:...|         Oakla

#### .take()

This returns the first `n` rows of the DataFrame as a list of Row objects. It is an action that triggers the execution of the DataFrame transformations and returns the results to the driver program.

In [102]:
trips.take(5)

[Row(ride_id='172DBBFC733F03CE', rideable_type='electric_bike', started_at=datetime.datetime(2024, 10, 10, 14, 54, 24, 572000), ended_at=datetime.datetime(2024, 10, 10, 15, 4, 7, 657000), start_station_name='Oakland Ave', start_station_id='JC022', end_station_name='Stevens - River Ter & 6 St', end_station_id='HB602', start_lat=40.7376037, start_lng=-74.0524783, end_lat=40.74313282710551, end_lng=-74.02698867022991, member_casual='member', rideDurationMinutes=9.72, rideDurationHours=0.16, rideDistanceKM=2.23, rideDistanceM=2230.0),
 Row(ride_id='D20BBA4860FE736C', rideable_type='electric_bike', started_at=datetime.datetime(2024, 10, 3, 19, 20, 21, 215000), ended_at=datetime.datetime(2024, 10, 3, 19, 31, 46, 511000), start_station_name='Oakland Ave', start_station_id='JC022', end_station_name='Stevens - River Ter & 6 St', end_station_id='HB602', start_lat=40.7376037, start_lng=-74.0524783, end_lat=40.74313282710551, end_lng=-74.02698867022991, member_casual='casual', rideDurationMinutes=

#### .show()

This method displays the top `n` rows of the DataFrame in a tabular format. It is an action that triggers the execution of the DataFrame transformations and returns the results to the driver program.

In [103]:
trips.show(5)

+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|172DBBFC733F03CE|electric_bike|2024-10-10 14:54:...|2024-10-10 15:04:...|       Oakland Ave| 

#### .drop()

This method removes a specified column or multiple columns from the DataFrame. It is a transformation that returns a new DataFrame without the dropped column(s).

In [104]:
df_dropped1 = trips.drop(col("end_station_name"))

df_dropped1.show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|172DBBFC733F03CE|electric_bike|2024-10-10 14:54:...|2024-10-10 15:04:...|         Oakland Ave|           JC022|         HB602|       40.7376037|      

In [105]:
cols_to_drop = ["start_station_name", "end_station_name"]

df_dropped2 = trips.drop(*cols_to_drop)

df_dropped2.show()

+----------------+-------------+--------------------+--------------------+----------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|start_station_id|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+--------------------+--------------------+----------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|172DBBFC733F03CE|electric_bike|2024-10-10 14:54:...|2024-10-10 15:04:...|           JC022|         HB602|       40.7376037|       -74.0524783|40.74313282710551|-74.02698867022991|       member|               9.72|

#### .dropDuplicates()

This method removes duplicate rows from the DataFrame based on all columns. It is a transformation that returns a new DataFrame without the duplicate rows.

In [106]:
trips_unique = trips.dropDuplicates()

print(f"Number of rides before: {trips.count()} | rides after: {trips_unique.count()}")

Number of rides before: 4689909 | rides after: 4689909


We can also drop duplicates based on a subset of columns by passing the column names as arguments to the `dropDuplicates()` method.

In [107]:
trips_unque2 = trips.dropDuplicates(["ride_id"])

print(f"Number of rides before: {trips.count()} | rides after: {trips_unque2.count()}")

Number of rides before: 4689909 | rides after: 4689886


In [108]:
trips.distinct().show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+------------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|         start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+------------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+
|C54B2D6A33C46C46|electric_bike|2024-10-22 17:21:...|2024-10-22 17:24:...|        Man

#### Union & Union Byname

`Union` combines two DataFrames with the same schema into a single DataFrame, while `unionByName` allows for combining DataFrames with different schemas by matching columns by name.

The syntax for `union` is as follows:

```python
df1.union(df2)
```

The syntax for `unionByName` is as follows:

```python
df1.unionByName(df2)
```

In [109]:
data1 = [("1", "kad"), ("2", "gad")]

schema1 = """ id string, name string"""

df1 = spark.createDataFrame(data1, schema1)

data2 = [("3", "med"), ("4", "had")]

schema2 = """ id string, name string"""

df2 = spark.createDataFrame(data2, schema2)

In [110]:
df1.show()

+---+----+
| id|name|
+---+----+
|  1| kad|
|  2| gad|
+---+----+



In [111]:
df2.show()

+---+----+
| id|name|
+---+----+
|  3| med|
|  4| had|
+---+----+



In [112]:
df_combined = df1.union(df2)

df_combined.show()

+---+----+
| id|name|
+---+----+
|  1| kad|
|  2| gad|
|  3| med|
|  4| had|
+---+----+



In [113]:
data3 = [("kad", "1"), ("gad", "2")]

schema3 = """ name string, id string"""

df3 = spark.createDataFrame(data3, schema3)

df_combined_2 = df2.unionByName(df3)

df_combined_2.show()

+---+----+
| id|name|
+---+----+
|  3| med|
|  4| had|
|  1| kad|
|  2| gad|
+---+----+



### String Functions

Some commonly used string functions in PySpark include: `initcap()`, `lower()`, `upper()`, `ltrim()`, `rtrim()`, `trim()`, and `concat_ws()`.

#### initcap()
The `initcap()` function capitalizes the first letter of each word in a string column.

In [114]:
trips.select(initcap("start_station_name").alias("start_station_name")).show()

+--------------------+
|  start_station_name|
+--------------------+
|         Oakland Ave|
|         Oakland Ave|
|         Oakland Ave|
|         Oakland Ave|
|     Brunswick & 6th|
|         Oakland Ave|
|      Pershing Field|
|      Pershing Field|
|City Hall - Washi...|
|City Hall - Washi...|
|City Hall - Washi...|
|City Hall - Washi...|
|         Exchange Pl|
|City Hall - Washi...|
|City Hall - Washi...|
|City Hall - Washi...|
|City Hall - Washi...|
|      Pershing Field|
|      Pershing Field|
|      Pershing Field|
+--------------------+
only showing top 20 rows


#### lower()

The `lower()` function converts all characters in a string column to lowercase.

In [115]:
trips.select(lower("start_station_name").alias("start_station_name")).show()

+--------------------+
|  start_station_name|
+--------------------+
|         oakland ave|
|         oakland ave|
|         oakland ave|
|         oakland ave|
|     brunswick & 6th|
|         oakland ave|
|      pershing field|
|      pershing field|
|city hall - washi...|
|city hall - washi...|
|city hall - washi...|
|city hall - washi...|
|         exchange pl|
|city hall - washi...|
|city hall - washi...|
|city hall - washi...|
|city hall - washi...|
|      pershing field|
|      pershing field|
|      pershing field|
+--------------------+
only showing top 20 rows


### Date Functions

Some commonly used date functions in PySpark include: `current_date()`, `current_timestamp()`, `date_add()`, `date_sub()`, `datediff()`, and `months_between()`.

#### .current_date()

The `current_date()` function returns the current date as a column of type DateType.

In [116]:
trips_updated = trips.withColumn("current_date", current_date())

trips_updated.show(1)

+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+----------+-----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+------------+
|         ride_id|rideable_type|          started_at|            ended_at|start_station_name|start_station_id|    end_station_name|end_station_id| start_lat|  start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|current_date|
+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+----------+-----------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+------------+
|172DBBFC733F03CE|electric_bike|2024-10-10 14:54:...|2024-10-10 15:04:...|       Oakland Ave|    

#### date_add()

The `date_add()` function adds a specified number of days to a date column and returns the resulting date.

In [117]:
trips_updated = trips_updated.withColumn("future_date", date_add("current_date", 7))

trips_updated.show(5)

+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+------------+-----------+
|         ride_id|rideable_type|          started_at|            ended_at|start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|current_date|future_date|
+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+------------+-----------+
|172DBBFC733F03CE|e

#### date_sub()

The `date_sub()` function subtracts a specified number of days from a date column and returns the resulting date.

In [118]:
trips_updated = trips_updated.withColumn("week_before", date_sub("current_date", 7))

trips_updated.show(5)

+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+------------+-----------+-----------+
|         ride_id|rideable_type|          started_at|            ended_at|start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|current_date|future_date|week_before|
+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+------------+--------

In [119]:
trips_updated = trips_updated.withColumn("week_before", date_add("current_date", -7))

trips_updated.show(5)

+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+------------+-----------+-----------+
|         ride_id|rideable_type|          started_at|            ended_at|start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|current_date|future_date|week_before|
+----------------+-------------+--------------------+--------------------+------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+------------+--------

#### .datediff()

The `datediff()` function calculates the difference in days between two date columns and returns the result as an integer column.

In [120]:
trips_updated = trips_updated.withColumn(
    "days",
    date_diff("current_date", "week_before"),
)

trips_updated.show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+------------+-----------+-----------+----+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|current_date|future_date|week_before|days|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+-----

#### .date_format()

In [ ]:
trips_updated = trips_updated.withColumn(
    "week_before", date_format("week_before", "dd-MM-yyyy")
)

trips_updated.show()

+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------------+-------------+------------+-----------+-----------+----+------------+
|         ride_id|rideable_type|          started_at|            ended_at|  start_station_name|start_station_id|    end_station_name|end_station_id|        start_lat|         start_lng|          end_lat|           end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|current_date|future_date|week_before|days|week_before_|
+----------------+-------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------+-----------------+------------------+-----------------+------------------+-------------+-------------------+-----------------+--------


#### .fillna()

In [123]:
trips_updated.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in trips_updated.columns]
).show()

+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+-------------------+-----------------+--------------+-------------+------------+-----------+-----------+----+------------+
|ride_id|rideable_type|started_at|ended_at|start_station_name|start_station_id|end_station_name|end_station_id|start_lat|start_lng|end_lat|end_lng|member_casual|rideDurationMinutes|rideDurationHours|rideDistanceKM|rideDistanceM|current_date|future_date|week_before|days|week_before_|
+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+-------------+-------------------+-----------------+--------------+-------------+------------+-----------+-----------+----+------------+
|      0|            0|         0|       0|               200|             200|            8379|          8813|        0|        0|      0|      0| 

There are no missing values in the DataFrame, so the `fillna()` function is not applicable in this case.

However, we can use the `fillna()` function to fill missing values in a DataFrame with a specified value. For example, we can fill all null values in a column with a default value or fill null values in multiple columns with different default values.

```python
df.fillna({'column1': 'default_value1', 'column2': 'default_value2'})
```

Or, to the entire DataFrame with a single value:

```python
df.fillna('default_value')
```

### Split & Indexing

Split and Indexing are common operations in PySpark for manipulating string columns and accessing specific elements within them. The `split()` function can be used to split a string column into an array of substrings based on a specified delimiter. 

After splitting, you can use indexing to access specific elements of the resulting array.

The syntax for the `split()` function is as follows:

```python
from pyspark.sql.functions import split

df.withColumn('new_column', split(df['string_column'], 'delimiter'))
```